In [1]:
%pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

  Using cached python_woc-0.4.1.tar.gz (483 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pandas-3.0.6-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.11.2-cp311-cp311-win_amd64.whl.metadata (80 kB)
  Using cached chardet-5.2.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_lzf-0.2.6.tar.gz (9.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached rapidgzip-0.14.5-c

  error: subprocess-exited-with-error
  
  exit code: 1
  
  [30 lines of output]
  Compiling C:\Users\Goodt\AppData\Local\Temp\pip-install-bb14xbfo\python-woc_defd345e54ef40f6a361201ac8d30bcc\woc\local.pyx because it changed.
  Compiling C:\Users\Goodt\AppData\Local\Temp\pip-install-bb14xbfo\python-woc_defd345e54ef40f6a361201ac8d30bcc\woc\tch.pyx because it changed.
  [1/2] Cythonizing C:\Users\Goodt\AppData\Local\Temp\pip-install-bb14xbfo\python-woc_defd345e54ef40f6a361201ac8d30bcc\woc\local.pyx
  [2/2] Cythonizing C:\Users\Goodt\AppData\Local\Temp\pip-install-bb14xbfo\python-woc_defd345e54ef40f6a361201ac8d30bcc\woc\tch.pyx
  running build_ext
  building 'woc.local' extension
  error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
  Traceback (most recent call last):
    File "C:\Users\Goodt\OneDrive - University of Tennessee\CS445\MiniProject2\.venv\Lib\site-packages\pip\_vendor\py

In [6]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['ViDA-NYU/reprozip', 'LLNL/mgmol', 'scifio/scifio-imageio', 'smistad/FAST', 
            'MouseLand/Kilosort', 'martini-alessandro/Maximum-Entropy-Spectrum', 'ShixiangWang/sigminer',
             'WorldWideTelescope/wwt-website', 'Q2MM/q2mm', 'natverse/nat.nblast']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

ModuleNotFoundError: No module named 'woc'

In [9]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
#import time
# let us first split df['sha1'] in chunks
#chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
#commit_data = []

#new
import time
import httpx
import json
from pathlib import Path

# Preserve existing results; recover from disk if the kernel was restarted.
if "commit_data" not in globals():
    checkpoint = Path("commit_data_checkpoint.json")
    commit_data = (
        json.loads(checkpoint.read_text()) if checkpoint.exists() else []
    )

# Only request commits we haven't successfully downloaded.
done = {record["commit"] for record in commit_data}
remaining = df.loc[~df["sha1"].isin(done), "sha1"].drop_duplicates()

chunks = [
    remaining.iloc[i:i + 10]
    for i in range(0, len(remaining), 10)
]

print(f"{len(done)} records retained; {len(remaining)} commits left to request.")

#old

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
    
    #res, err = woc.get_values_many('commit.tch',chunk.to_list())
    for attempt in range(5):
        try:
            res, err = woc.get_values_many("commit.tch", chunk.to_list())
            break
        except (httpx.TimeoutException, httpx.NetworkError):
            if attempt == 4:
                with open("commit_data_checkpoint.json", "w") as f:
                    json.dump(commit_data, f)
                print("Repeated network failures. Progress saved; rerun later.")
                raise

            delay = 5 * (2 ** attempt)
            print(f"Network interruption. Retrying in {delay} seconds...")
            time.sleep(delay)

    res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
    time.sleep(1.5)

    if err: # check for errors
        print('Got Errors', err)

    for commit_sha, commit in res.items():
    # flatten commit objects
        commit_data.append({
            'commit': commit_sha,
            'tree': commit[0],
            'parent': list(commit[1]),
            'author': commit[2][0],
            'author_time': int(commit[2][1]),
            'author_tz': commit[2][2],
            'committer': commit[3][0],
            'committer_time': int(commit[3][1]),
            'committer_tz': commit[3][2],
            'message': commit[4],
        })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

4893 records retained; 10592 commits left to request.


  0%|                                                               | 1/1060 [00:01<29:57,  1.70s/it]

Got Errors {'0072c7cfe4772a5b831cb9fd67d4e9413d29725a': 'Key 0072c7cfe4772a5b831cb9fd67d4e9413d29725a not found in /da5_fast/All.sha1c/commit_0.tch', '041e80fa39ad5893d3df3e12807f3f120812e109': 'Key 041e80fa39ad5893d3df3e12807f3f120812e109 not found in /da5_fast/All.sha1c/commit_4.tch', '066adb3fcfa459bbe2ef073416dac112d1e61f35': 'Key 066adb3fcfa459bbe2ef073416dac112d1e61f35 not found in /da5_fast/All.sha1c/commit_6.tch', '070a66d2b9631db231dabfd08fef9831cf10081f': 'Key 070a66d2b9631db231dabfd08fef9831cf10081f not found in /da5_fast/All.sha1c/commit_7.tch', '082bb44f6ba5196241de24427b41d3cc99fbaadd': 'Key 082bb44f6ba5196241de24427b41d3cc99fbaadd not found in /da5_fast/All.sha1c/commit_8.tch', '09442a12d33f0bf2344507c91af38d7a118435d9': 'Key 09442a12d33f0bf2344507c91af38d7a118435d9 not found in /da5_fast/All.sha1c/commit_9.tch', '0a9034f3590b7b2dfa33831e7c35bb5efeda502c': 'Key 0a9034f3590b7b2dfa33831e7c35bb5efeda502c not found in /da5_fast/All.sha1c/commit_10.tch', '0df60864ff06c16684fa

  0%|                                                               | 2/1060 [00:03<28:35,  1.62s/it]

Got Errors {'1710a7003d6fa09821c903721a9b822fb279a0cc': 'Key 1710a7003d6fa09821c903721a9b822fb279a0cc not found in /da5_fast/All.sha1c/commit_23.tch', '171980fb03021ae66c1b9571f4ee26723d863642': 'Key 171980fb03021ae66c1b9571f4ee26723d863642 not found in /da5_fast/All.sha1c/commit_23.tch', '185164dcb67b380ea62513428525b1cad49aec01': 'Key 185164dcb67b380ea62513428525b1cad49aec01 not found in /da5_fast/All.sha1c/commit_24.tch', '1b96a1533c530e4088807ad160dc8605eb48bf2b': 'Key 1b96a1533c530e4088807ad160dc8605eb48bf2b not found in /da5_fast/All.sha1c/commit_27.tch', '1c776a58fdc0485650a169b1fd1b2a437bc955c1': 'Key 1c776a58fdc0485650a169b1fd1b2a437bc955c1 not found in /da5_fast/All.sha1c/commit_28.tch', '279f09862468c969d0fb31a736db93a2b1228fc5': 'Key 279f09862468c969d0fb31a736db93a2b1228fc5 not found in /da5_fast/All.sha1c/commit_39.tch', '282d8908ffb903926e3f5d8561895dc699fe086a': 'Key 282d8908ffb903926e3f5d8561895dc699fe086a not found in /da5_fast/All.sha1c/commit_40.tch', '2a324f79464686

...

Got Errors {'52092b155a52056ac5b90bb59f293ff1f45d449d': 'Key 52092b155a52056ac5b90bb59f293ff1f45d449d not found in /da5_fast/All.sha1c/commit_82.tch'}


 99%|███████████████████████████████████████████████████████████▏| 1045/1060 [27:40<00:23,  1.59s/it]

Got Errors {'aab65b1241994fc5676d01b4ea411ac8e0314ce5': 'Key aab65b1241994fc5676d01b4ea411ac8e0314ce5 not found in /da5_fast/All.sha1c/commit_42.tch'}


100%|████████████████████████████████████████████████████████████| 1060/1060 [28:04<00:00,  1.59s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00092a4323f35b8e7489bc57d03cc2a798cdd057,0c069bb9fcb56daaf203c76c6bb10a9e68292bb4,[f3698af18b46fd599197576b385d7bcc2bdde4bf],Remi Rampin <remirampin@gmail.com>,1403040310,-0400,Remi Rampin <remirampin@gmail.com>,1403040310,-0400,Fixes TracedFile getting created and never upd...,00092a4323f35b8e7489bc57d03cc2a798cdd057,vida-nyu_reprozip
1,001df9285761c6595f20aefb207446b96d48f80c,07eb076f73ec3bfa9304921a0edf8c018bac6859,[3bc3d28627d7c244bdf1600afa81e464ce3e60d8],Remi Rampin <remirampin@gmail.com>,1476041829,-0400,Remi Rampin <remirampin@gmail.com>,1476041829,-0400,GUI: Confirm exit if still set up\n,001df9285761c6595f20aefb207446b96d48f80c,vida-nyu_reprozip


In [7]:
import json

with open("commit_data_checkpoint.json", "w") as f:
    json.dump(commit_data, f)

print(f"Saved {len(commit_data)} commit records.")

Saved 4893 commit records.


In [10]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00092a4323f35b8e7489bc57d03cc2a798cdd057', 'tree': '0c069bb9fcb56daaf203c76c6bb10a9e68292bb4', 'parent': ['f3698af18b46fd599197576b385d7bcc2bdde4bf'], 'author': 'Remi Rampin <remirampin@gmail.com>', 'author_time': 1403040310, 'author_tz': '-0400', 'committer': 'Remi Rampin <remirampin@gmail.com>', 'committer_time': 1403040310, 'committer_tz': '-0400', 'message': "Fixes TracedFile getting created and never updated\n\nProbably doesn't change much.\n"}


In [4]:
import pandas as pd
projects = ['ViDA-NYU/reprozip', 'LLNL/mgmol', 'scifio/scifio-imageio', 'smistad/FAST', 
#            'MouseLand/Kilosort', 'martini-alessandro/Maximum-Entropy-Spectrum', 'ShixiangWang/sigminer',
#             'WorldWideTelescope/wwt-website', 'Q2MM/q2mm', 'natverse/nat.nblast']
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for prj in projects:
    for k in commit_data:
        row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
        dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


NameError: name 'commit_data' is not defined

In [12]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,natverse_nat.nblast,00092a4323f35b8e7489bc57d03cc2a798cdd057,Remi Rampin <remirampin@gmail.com>,1403040310,Fixes TracedFile getting created and never upd...


In [13]:
#mode a means append, so you have all your projects in the same file
yournetid='ggood3'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

[number of stars, number of forks, last commit date]

1. ViDA-NYU/reprozip:
    [363, 37, 01/18/2026]
2. LLNL/mgmol:
    [47, 15, 09/11/2026]
3. scifio/scifio-imageio:
    [16, 14, 04/22/2026]
4. smistad/FAST:
    [517, 113, 09/16/2026]
5. MouseLand/Kilosort:
    [630, 295, 09/25/2026]
6. martini-alessandro/Maximum-Entropy-Spectrum:
    [48, 11, 11/18/2024]
7. ShixiangWang/sigminer:
    [163, 20, 09/26/2026]
8. WorldWideTelescope/wwt-website:
    [44, 18, 06/10/2026]
9. Q2MM/q2mm:
    [25, 9, 04/20/2026]
10. natverse/nat.nblast:
    [18, 7, 08/24/2026]

| Project | Number of commits | Number of authors | Max time (UTC) | Min time (UTC) |
|---|---:|---:|---|---|
| llnl_mgmol | 1692 | 21 | 2025-11-04 01:42:17 | 2017-07-06 02:49:42 |
| martini-alessandro_maximum-entropy-spectrum | 405 | 9 | 2024-11-18 17:21:19 | 2020-12-09 20:34:54 |
| mouseland_kilosort | 1822 | 72 | 2025-11-04 22:38:00 | 2018-05-04 12:29:09 |
| natverse_nat.nblast | 419 | 8 | 2025-08-18 11:32:59 | 2014-04-03 18:19:21 |
| q2mm_q2mm | 869 | 27 | 2025-08-05 20:37:04 | 2014-08-26 00:22:16 |
| scifio_scifio-imageio | 322 | 31 | 2025-06-27 13:28:13 | 2011-03-16 18:13:38 |
| shixiangwang_sigminer | 1296 | 16 | 2025-12-25 15:46:11 | 2018-12-29 09:28:51 |
| smistad_fast | 4450 | 34 | 2025-11-06 14:57:05 | 2014-03-19 11:02:11 |
| vida-nyu_reprozip | 3244 | 45 | 2026-02-04 04:07:01 | 2014-05-02 21:41:32 |
| worldwidetelescope_wwt-website | 886 | 21 | 2025-07-30 13:29:47 | 2015-05-01 02:46:39 |
